In [0]:
# Criar tabela silver_business_amostra no schema yelp_ing
# Filtrar food_category diferente de "OUTROS" e limitar a 1000 registros
df_business_amostra = spark.table("yelp_ing.silver_business") \
    .filter("food_category != 'OUTROS'") \
    .limit(1000)

# Salvar como tabela no schema yelp_ing
df_business_amostra.write.mode("overwrite").saveAsTable("yelp_ing.silver_business_amostra")

# Mostrar a tabela criada
print("Tabela yelp_ing.silver_business_amostra criada com sucesso!")
display(spark.table("yelp_ing.silver_business_amostra"))

print("\n" + "="*80 + "\n")

# Criar tabela silver_reviews_filtered_amostra no schema yelp_ing
# Limitar a 1000 registros
df_reviews_amostra = spark.table("yelp_ing.silver_review_filtered") \
    .limit(1000)

# Salvar como tabela no schema yelp_ing
df_reviews_amostra.write.mode("overwrite").saveAsTable("yelp_ing.silver_reviews_filtered_amostra")

# Mostrar a tabela criada
print("Tabela yelp_ing.silver_reviews_filtered_amostra criada com sucesso!")
display(spark.table("yelp_ing.silver_reviews_filtered_amostra"))

In [0]:
spark.sql("select count(*) from yelp_ing.silver_business_amostra").display()
spark.sql("select count(*) from yelp_ing.silver_reviews_filtered_amostra").display()

In [0]:
# ============================================================================
# ETAPA 1A - PREPARAÇÃO DOS DADOS: REVIEWS (SILVER REFINADA)
# ============================================================================

from pyspark.sql.functions import (
    to_timestamp, date_format, hour, dayofweek, 
    col, when, current_timestamp
)

print("Preparando dados de Reviews...")
print("=" * 80)

# Carregar tabela de reviews
df_reviews = spark.table("yelp_ing.silver_reviews_filtered_amostra")

print(f"Total de reviews carregados: {df_reviews.count():,}")

# Criar colunas adicionais
df_reviews_refined = df_reviews \
    .withColumn("review_timestamp", to_timestamp(col("date"), "yyyy-MM-dd HH:mm:ss")) \
    .withColumn("review_date", to_timestamp(col("date"), "yyyy-MM-dd HH:mm:ss").cast("date")) \
    .withColumn("review_hour", hour(to_timestamp(col("date"), "yyyy-MM-dd HH:mm:ss"))) \
    .withColumn("review_day_of_week_num", dayofweek(to_timestamp(col("date"), "yyyy-MM-dd HH:mm:ss"))) \
    .withColumn("review_day_of_week", 
        when(col("review_day_of_week_num") == 1, "Sunday")
        .when(col("review_day_of_week_num") == 2, "Monday")
        .when(col("review_day_of_week_num") == 3, "Tuesday")
        .when(col("review_day_of_week_num") == 4, "Wednesday")
        .when(col("review_day_of_week_num") == 5, "Thursday")
        .when(col("review_day_of_week_num") == 6, "Friday")
        .when(col("review_day_of_week_num") == 7, "Saturday")
    )

# Adicionar metadados de processamento
df_reviews_refined = df_reviews_refined.withColumn(
    "data_processamento_gold",
    current_timestamp()
)

print(f"\nColunas adicionadas:")
print("  - review_timestamp (timestamp)")
print("  - review_date (date)")
print("  - review_hour (int)")
print("  - review_day_of_week (string)")
print("  - review_day_of_week_num (int)")

# Salvar como tabela Silver refinada
table_name = "yelp_ing.silver_reviews_refined"
df_reviews_refined.write.mode("overwrite").saveAsTable(table_name)

print(f"\n✓ Tabela criada: {table_name}")
print(f"  Total de registros: {df_reviews_refined.count():,}")
print(f"  Total de colunas: {len(df_reviews_refined.columns)}")

# Mostrar amostra
print("\nAmostra dos dados:")
display(df_reviews_refined.select(
    "review_id", "business_id", "user_id", "date", 
    "review_timestamp", "review_date", "review_hour", "review_day_of_week"
).limit(5))

In [0]:
# ============================================================================
# ETAPA 1B - PREPARAÇÃO DOS DADOS: BUSINESS (SILVER REFINADA)
# ============================================================================

from pyspark.sql.functions import (
    col, explode, array, struct, lit, split, regexp_replace, when, current_timestamp
)

print("Preparando dados de Business...")
print("=" * 80)

# Carregar tabela de business
df_business = spark.table("yelp_ing.silver_business_amostra")

print(f"Total de estabelecimentos carregados: {df_business.count():,}")

# Explodir o campo 'hours' (struct) em linhas por dia da semana
df_business_exploded = df_business.select(
    "business_id",
    "name",
    "is_open",
    "food_category",
    "city",
    "state",
    explode(array(
        struct(lit("Monday").alias("day"), col("hours.Monday").alias("hours_str")),
        struct(lit("Tuesday").alias("day"), col("hours.Tuesday").alias("hours_str")),
        struct(lit("Wednesday").alias("day"), col("hours.Wednesday").alias("hours_str")),
        struct(lit("Thursday").alias("day"), col("hours.Thursday").alias("hours_str")),
        struct(lit("Friday").alias("day"), col("hours.Friday").alias("hours_str")),
        struct(lit("Saturday").alias("day"), col("hours.Saturday").alias("hours_str")),
        struct(lit("Sunday").alias("day"), col("hours.Sunday").alias("hours_str"))
    )).alias("day_hours")
).select(
    "business_id",
    "name",
    "is_open",
    "food_category",
    "city",
    "state",
    col("day_hours.day").alias("business_day_of_week"),
    col("day_hours.hours_str").alias("hours_str")
)

# Parsear o campo hours_str (formato: "11:0-22:0" ou "0:0-0:0")
# Extrair open_time e close_time
df_business_refined = df_business_exploded \
    .withColumn("open_time", 
        when(col("hours_str").isNotNull(), 
            regexp_replace(split(col("hours_str"), "-")[0], ":", "")
        )
    ) \
    .withColumn("close_time", 
        when(col("hours_str").isNotNull(), 
            regexp_replace(split(col("hours_str"), "-")[1], ":", "")
        )
    ) \
    .withColumn("is_closed_day",
        when(col("hours_str").isNull(), 1)
        .when(col("hours_str") == "0:0-0:0", 1)
        .otherwise(0)
    ) \
    .withColumn("data_processamento_gold", current_timestamp())

print(f"\nColunas adicionadas:")
print("  - business_day_of_week (string)")
print("  - open_time (string - formato HHMM)")
print("  - close_time (string - formato HHMM)")
print("  - is_closed_day (int - 1 se fechado naquele dia)")

# Salvar como tabela Silver refinada
table_name = "yelp_ing.silver_business_refined"
df_business_refined.write.mode("overwrite").saveAsTable(table_name)

print(f"\n✓ Tabela criada: {table_name}")
print(f"  Total de registros: {df_business_refined.count():,}")
print(f"  Total de colunas: {len(df_business_refined.columns)}")

# Mostrar amostra
print("\nAmostra dos dados:")
display(df_business_refined.select(
    "business_id", "name", "is_open", "business_day_of_week", 
    "hours_str", "open_time", "close_time", "is_closed_day"
).limit(10))

In [0]:
# ============================================================================
# ETAPA 2 - CRUZAMENTO E DETECÇÃO DE FRAUDE
# ============================================================================

from pyspark.sql.functions import col, when, lpad, concat, lit, current_timestamp

print("Realizando cruzamento Reviews x Business e detectando fraudes...")
print("=" * 80)

# Carregar tabelas refinadas
df_reviews = spark.table("yelp_ing.silver_reviews_refined")
df_business = spark.table("yelp_ing.silver_business_refined")

print(f"Reviews carregados: {df_reviews.count():,}")
print(f"Business registros carregados: {df_business.count():,}")

# Join por business_id e dia da semana
df_joined = df_reviews.join(
    df_business,
    (df_reviews.business_id == df_business.business_id) &
    (df_reviews.review_day_of_week == df_business.business_day_of_week),
    "left"
)

print(f"\nRegistros após join: {df_joined.count():,}")

# Converter review_hour para formato HHMM (padded com zeros)
df_joined = df_joined.withColumn(
    "review_hour_str",
    lpad(concat(col("review_hour").cast("string"), lit("00")), 4, "0")
)

# LÓGICA DE DETECÇÃO DE FRAUDE
# flag_possivel_fraude = 1 se:
# 1. Estabelecimento está permanentemente fechado (is_open = 0)
# 2. Review foi feita em dia que o estabelecimento não abre (is_closed_day = 1)
# 3. Review feita fora do horário de funcionamento

df_fraud_base = df_joined.withColumn(
    "flag_possivel_fraude",
    when(
        # Estabelecimento permanentemente fechado
        col("is_open") == 0, 1
    ).when(
        # Dia que não abre
        col("is_closed_day") == 1, 1
    ).when(
        # Fora do horário (review antes de abrir ou depois de fechar)
        (col("open_time").isNotNull()) & (col("close_time").isNotNull()) &
        ((col("review_hour_str") < col("open_time")) | (col("review_hour_str") > col("close_time"))),
        1
    ).when(
        # Sem informação de horário (não podemos validar)
        col("open_time").isNull(), 0
    ).otherwise(0)
)

# Selecionar colunas finais
df_fraud_base_final = df_fraud_base.select(
    df_reviews.business_id.alias("business_id"),
    df_reviews.review_id,
    df_reviews.user_id,
    df_reviews.stars,
    df_reviews.review_date,
    df_reviews.review_timestamp,
    df_reviews.review_hour,
    df_reviews.review_day_of_week,
    df_business.name.alias("business_name"),
    df_business.food_category,
    df_business.city,
    df_business.state,
    df_business.is_open,
    df_business.business_day_of_week,
    df_business.open_time,
    df_business.close_time,
    df_business.is_closed_day,
    "flag_possivel_fraude"
).withColumn("data_processamento_gold", current_timestamp())

# Estatísticas
total_reviews = df_fraud_base_final.count()
total_fraud = df_fraud_base_final.filter(col("flag_possivel_fraude") == 1).count()
total_valid = total_reviews - total_fraud

print(f"\n=" * 80)
print("ESTATÍSTICAS DE FRAUDE:")
print(f"  Total de reviews: {total_reviews:,}")
print(f"  Reviews com fraude: {total_fraud:,} ({total_fraud/total_reviews*100:.2f}%)")
print(f"  Reviews válidas: {total_valid:,} ({total_valid/total_reviews*100:.2f}%)")
print(f"=" * 80)

# Salvar como tabela Gold
table_name = "yelp_ing.gold_review_fraud_base"
df_fraud_base_final.write.mode("overwrite").saveAsTable(table_name)

print(f"\n✓ Tabela Gold criada: {table_name}")

# Mostrar amostra de fraudes detectadas
print("\nAmostra de reviews com possível fraude:")
display(df_fraud_base_final.filter(col("flag_possivel_fraude") == 1).limit(10))

In [0]:
# ============================================================================
# ETAPA 3 - AGREGAÇÃO POR USUÁRIO
# ============================================================================

from pyspark.sql.functions import col, count, sum, when, concat, lit, current_timestamp

print("Agregando dados por usuário...")
print("=" * 80)

# Carregar base de fraude
df_fraud_base = spark.table("yelp_ing.gold_review_fraud_base")

# Agregar por user_id
df_user_summary = df_fraud_base.groupBy("user_id").agg(
    count("review_id").alias("total_reviews"),
    sum(when(col("flag_possivel_fraude") == 1, 1).otherwise(0)).alias("total_reviews_fraude")
)

# Criar mensagem de alerta
df_user_summary = df_user_summary.withColumn(
    "mensagem_alerta_user",
    when(
        col("total_reviews_fraude") > 0,
        concat(
            lit("User "),
            col("user_id"),
            lit(" realizou "),
            col("total_reviews_fraude").cast("string"),
            lit(" reviews em dias/horários em que o respectivo estabelecimento estava fechado")
        )
    ).otherwise(lit("Nenhuma fraude detectada"))
).withColumn("data_processamento_gold", current_timestamp())

# Estatísticas
total_users = df_user_summary.count()
users_with_fraud = df_user_summary.filter(col("total_reviews_fraude") > 0).count()

print(f"\n=" * 80)
print("ESTATÍSTICAS POR USUÁRIO:")
print(f"  Total de usuários: {total_users:,}")
print(f"  Usuários com fraude: {users_with_fraud:,} ({users_with_fraud/total_users*100:.2f}%)")
print(f"=" * 80)

# Salvar como tabela Gold
table_name = "yelp_ing.gold_user_fraud_summary"
df_user_summary.write.mode("overwrite").saveAsTable(table_name)

print(f"\n✓ Tabela Gold criada: {table_name}")

# Mostrar usuários com mais fraudes
print("\nTop 10 usuários com mais reviews fraudulentas:")
display(df_user_summary.filter(col("total_reviews_fraude") > 0).orderBy(col("total_reviews_fraude").desc()).limit(10))

In [0]:
# ============================================================================
# ETAPA 4 - AGREGAÇÃO POR ESTABELECIMENTO
# ============================================================================

from pyspark.sql.functions import (
    col, count, sum, when, concat, lit, countDistinct, 
    collect_set, array_join, current_timestamp
)
from pyspark.sql.window import Window

print("Agregando dados por estabelecimento...")
print("=" * 80)

# Carregar base de fraude
df_fraud_base = spark.table("yelp_ing.gold_review_fraud_base")

# 1. Agregação básica por business_id
df_business_summary = df_fraud_base.groupBy("business_id", "business_name", "food_category").agg(
    count("review_id").alias("total_reviews"),
    sum(when(col("flag_possivel_fraude") == 1, 1).otherwise(0)).alias("total_reviews_fraude")
)

# 2. Detectar comportamento suspeito por usuário
# Contar reviews por user_id, business_id e data
df_user_behavior = df_fraud_base.groupBy("user_id", "business_id", "review_date", "food_category").agg(
    count("review_id").alias("reviews_same_day_same_business")
)

# Contar reviews do mesmo usuário em restaurantes no mesmo dia
df_user_behavior_restaurants = df_fraud_base \
    .filter(col("food_category") == "RESTAURANTE") \
    .groupBy("user_id", "review_date").agg(
        countDistinct("business_id").alias("restaurants_reviewed_same_day")
    )

# Identificar alertas de comportamento suspeito
df_suspicious_users = df_user_behavior \
    .filter(col("reviews_same_day_same_business") > 10) \
    .select("user_id", "business_id") \
    .distinct() \
    .withColumn(
        "alert_type",
        lit("same_business_many_reviews")
    )

df_suspicious_restaurants = df_user_behavior_restaurants \
    .filter(col("restaurants_reviewed_same_day") > 5) \
    .join(df_fraud_base.select("user_id", "business_id", "review_date").distinct(), ["user_id", "review_date"]) \
    .select("user_id", "business_id") \
    .distinct() \
    .withColumn(
        "alert_type",
        lit("many_restaurants_same_day")
    )

# Unir alertas
df_all_alerts = df_suspicious_users.union(df_suspicious_restaurants)

# Agregar alertas por business
df_alerts_by_business = df_all_alerts.groupBy("business_id").agg(
    collect_set(
        when(col("alert_type") == "same_business_many_reviews",
            concat(
                lit("User "),
                col("user_id"),
                lit(" realizou mais do que 10 reviews no mesmo estabelecimento no mesmo dia")
            )
        ).when(col("alert_type") == "many_restaurants_same_day",
            concat(
                lit("User "),
                col("user_id"),
                lit(" realizou mais do que 5 reviews em Restaurantes no mesmo dia")
            )
        )
    ).alias("user_alerts_list")
)

# Join com o sumário de business
df_business_final = df_business_summary.join(
    df_alerts_by_business,
    "business_id",
    "left"
)

# Criar mensagens de alerta
df_business_final = df_business_final \
    .withColumn(
        "mensagem_alerta_business",
        when(
            col("total_reviews_fraude") > 0,
            concat(
                lit("Estabelecimento "),
                col("business_name"),
                lit(" (ID: "),
                col("business_id"),
                lit(") recebeu "),
                col("total_reviews_fraude").cast("string"),
                lit(" reviews em dias/horários em que estava fechado")
            )
        ).otherwise(lit("Nenhuma fraude detectada"))
    ) \
    .withColumn(
        "mensagem_alerta_user_2",
        when(
            col("user_alerts_list").isNotNull(),
            array_join(col("user_alerts_list"), "; ")
        ).otherwise(lit("Nenhum comportamento suspeito detectado"))
    ) \
    .withColumn("data_processamento_gold", current_timestamp()) \
    .drop("user_alerts_list")

# Estatísticas
total_businesses = df_business_final.count()
businesses_with_fraud = df_business_final.filter(col("total_reviews_fraude") > 0).count()

print(f"\n=" * 80)
print("ESTATÍSTICAS POR ESTABELECIMENTO:")
print(f"  Total de estabelecimentos: {total_businesses:,}")
print(f"  Estabelecimentos com fraude: {businesses_with_fraud:,} ({businesses_with_fraud/total_businesses*100:.2f}%)")
print(f"=" * 80)

# Salvar como tabela Gold
table_name = "yelp_ing.gold_business_fraud_summary"
df_business_final.write.mode("overwrite").saveAsTable(table_name)

print(f"\n✓ Tabela Gold criada: {table_name}")

# Mostrar estabelecimentos com mais fraudes
print("\nTop 10 estabelecimentos com mais reviews fraudulentas:")
display(df_business_final.filter(col("total_reviews_fraude") > 0).orderBy(col("total_reviews_fraude").desc()).limit(10))

In [0]:
# ============================================================================
# ETAPA 5 - MÉTRICAS GERAIS
# ============================================================================

from pyspark.sql.functions import col, count, sum, when, lit, current_timestamp

print("Calculando métricas gerais...")
print("=" * 80)

# Carregar base de fraude
df_fraud_base = spark.table("yelp_ing.gold_review_fraud_base")

# Calcular métricas
total_reviews = df_fraud_base.count()
total_fraud = df_fraud_base.filter(col("flag_possivel_fraude") == 1).count()
total_valid = total_reviews - total_fraud

# Criar DataFrame com as métricas
df_metrics = spark.createDataFrame([
    (total_reviews, total_fraud, total_valid)
], ["total_reviews", "total_fraud_reviews", "total_valid_reviews"])

df_metrics = df_metrics.withColumn("data_processamento_gold", current_timestamp())

print(f"\n=" * 80)
print("MÉTRICAS GERAIS:")
print(f"  Total de reviews: {total_reviews:,}")
print(f"  Reviews com fraude: {total_fraud:,} ({total_fraud/total_reviews*100:.2f}%)")
print(f"  Reviews válidas: {total_valid:,} ({total_valid/total_reviews*100:.2f}%)")
print(f"=" * 80)

# Salvar como tabela Gold
table_name = "yelp_ing.gold_fraud_metrics"
df_metrics.write.mode("overwrite").saveAsTable(table_name)

print(f"\n✓ Tabela Gold criada: {table_name}")

# Mostrar métricas
print("\nMétricas:")
display(df_metrics)

In [0]:
# ============================================================================
# ETAPA 6 - DISTRIBUIÇÃO TEMPORAL POR FAIXA DE HORÁRIO
# ============================================================================

from pyspark.sql.functions import col, when, count, sum, current_timestamp

print("Calculando distribuição temporal por faixa de horário...")
print("=" * 80)

# Carregar base de fraude
df_fraud_base = spark.table("yelp_ing.gold_review_fraud_base")

# Criar buckets de horário
df_time_distribution = df_fraud_base.withColumn(
    "time_bucket",
    when((col("review_hour") >= 0) & (col("review_hour") < 3), "00:00-03:00")
    .when((col("review_hour") >= 3) & (col("review_hour") < 6), "03:01-06:00")
    .when((col("review_hour") >= 6) & (col("review_hour") < 9), "06:01-09:00")
    .when((col("review_hour") >= 9) & (col("review_hour") < 12), "09:01-12:00")
    .when((col("review_hour") >= 12) & (col("review_hour") < 15), "12:01-15:00")
    .when((col("review_hour") >= 15) & (col("review_hour") < 18), "15:01-18:00")
    .when((col("review_hour") >= 18) & (col("review_hour") < 21), "18:01-21:00")
    .when((col("review_hour") >= 21) & (col("review_hour") <= 23), "21:01-23:59")
    .otherwise("Unknown")
)

# Agregar por time_bucket
df_time_agg = df_time_distribution.groupBy("time_bucket").agg(
    count("review_id").alias("total_reviews"),
    sum(when(col("flag_possivel_fraude") == 1, 1).otherwise(0)).alias("total_fraud"),
    sum(when(col("flag_possivel_fraude") == 0, 1).otherwise(0)).alias("total_valid")
).withColumn("data_processamento_gold", current_timestamp())

# Ordenar por horário
df_time_agg = df_time_agg.orderBy(
    when(col("time_bucket") == "00:00-03:00", 1)
    .when(col("time_bucket") == "03:01-06:00", 2)
    .when(col("time_bucket") == "06:01-09:00", 3)
    .when(col("time_bucket") == "09:01-12:00", 4)
    .when(col("time_bucket") == "12:01-15:00", 5)
    .when(col("time_bucket") == "15:01-18:00", 6)
    .when(col("time_bucket") == "18:01-21:00", 7)
    .when(col("time_bucket") == "21:01-23:59", 8)
    .otherwise(9)
)

print(f"\n=" * 80)
print("DISTRIBUIÇÃO TEMPORAL:")
df_time_agg.show(truncate=False)
print(f"=" * 80)

# Salvar como tabela Gold
table_name = "yelp_ing.gold_reviews_time_distribution"
df_time_agg.write.mode("overwrite").saveAsTable(table_name)

print(f"\n✓ Tabela Gold criada: {table_name}")

# Mostrar distribuição
print("\nDistribuição temporal de reviews:")
display(df_time_agg)

In [0]:
# ============================================================================
# ETAPA 7 - QUERIES PARA DASHBOARD
# ============================================================================

print("=" * 80)
print("QUERIES SQL PRONTAS PARA DASHBOARD")
print("=" * 80)

queries = {
    "KPI - Métricas Gerais": """
        SELECT 
            total_reviews,
            total_fraud_reviews,
            total_valid_reviews,
            ROUND(total_fraud_reviews * 100.0 / total_reviews, 2) as fraud_percentage
        FROM yelp_ing.gold_fraud_metrics
    """,
    
    "Distribuição Temporal (Gráfico de Barras)": """
        SELECT 
            time_bucket,
            total_reviews,
            total_fraud,
            total_valid
        FROM yelp_ing.gold_reviews_time_distribution
        ORDER BY 
            CASE time_bucket
                WHEN '00:00-03:00' THEN 1
                WHEN '03:01-06:00' THEN 2
                WHEN '06:01-09:00' THEN 3
                WHEN '09:01-12:00' THEN 4
                WHEN '12:01-15:00' THEN 5
                WHEN '15:01-18:00' THEN 6
                WHEN '18:01-21:00' THEN 7
                WHEN '21:01-23:59' THEN 8
            END
    """,
    
    "Alertas de Usuários (Top 20)": """
        SELECT 
            user_id,
            total_reviews,
            total_reviews_fraude,
            mensagem_alerta_user
        FROM yelp_ing.gold_user_fraud_summary
        WHERE total_reviews_fraude > 0
        ORDER BY total_reviews_fraude DESC
        LIMIT 20
    """,
    
    "Alertas de Estabelecimentos (Top 20)": """
        SELECT 
            business_name,
            food_category,
            total_reviews,
            total_reviews_fraude,
            mensagem_alerta_business,
            mensagem_alerta_user_2
        FROM yelp_ing.gold_business_fraud_summary
        WHERE total_reviews_fraude > 0
        ORDER BY total_reviews_fraude DESC
        LIMIT 20
    """,
    
    "Reviews Fraudulentas Detalhadas": """
        SELECT 
            review_id,
            user_id,
            business_name,
            food_category,
            review_date,
            review_hour,
            review_day_of_week,
            is_open,
            open_time,
            close_time,
            flag_possivel_fraude
        FROM yelp_ing.gold_review_fraud_base
        WHERE flag_possivel_fraude = 1
        ORDER BY review_date DESC
        LIMIT 100
    """
}

for query_name, query_sql in queries.items():
    print(f"\n{'=' * 80}")
    print(f"QUERY: {query_name}")
    print(f"{'=' * 80}")
    print(query_sql.strip())
    print()

print("\n" + "=" * 80)
print("RESUMO DAS TABELAS CRIADAS")
print("=" * 80)

tabelas_criadas = [
    ("yelp_ing.silver_reviews_refined", "Reviews com colunas de data/hora extraídas"),
    ("yelp_ing.silver_business_refined", "Business com horários explodidos por dia da semana"),
    ("yelp_ing.gold_review_fraud_base", "Base completa com flag de fraude"),
    ("yelp_ing.gold_user_fraud_summary", "Agregação por usuário com alertas"),
    ("yelp_ing.gold_business_fraud_summary", "Agregação por estabelecimento com alertas"),
    ("yelp_ing.gold_fraud_metrics", "Métricas gerais (KPIs)"),
    ("yelp_ing.gold_reviews_time_distribution", "Distribuição temporal por faixa de horário")
]

print("\nTabelas Silver Refinada:")
for tabela, descricao in tabelas_criadas[:2]:
    print(f"  ✓ {tabela}")
    print(f"    {descricao}")

print("\nTabelas Gold:")
for tabela, descricao in tabelas_criadas[2:]:
    print(f"  ✓ {tabela}")
    print(f"    {descricao}")

print("\n" + "=" * 80)
print("✓ PIPELINE COMPLETO CRIADO COM SUCESSO!")
print("=" * 80)

In [0]:
# ============================================================================
# RESUMO EXECUTIVO - RESULTADOS DO PIPELINE DE DETECÇÃO DE FRAUDES
# ============================================================================

print("\n" + "=" * 80)
print("RESUMO EXECUTIVO - PIPELINE DE DETECÇÃO DE FRAUDES YELP")
print("=" * 80)

print("\n📊 ESTATÍSTICAS CONSOLIDADAS:")
print("-" * 80)

# Carregar métricas
df_metrics = spark.table("yelp_ing.gold_fraud_metrics")
metrics = df_metrics.collect()[0]

print(f"  Total de Reviews Analisados: {metrics['total_reviews']:,}")
print(f"  Reviews com Fraude Detectada: {metrics['total_fraud_reviews']:,} ({metrics['total_fraud_reviews']/metrics['total_reviews']*100:.2f}%)")
print(f"  Reviews Válidas: {metrics['total_valid_reviews']:,} ({metrics['total_valid_reviews']/metrics['total_reviews']*100:.2f}%)")

# Estatísticas por usuário
df_users = spark.table("yelp_ing.gold_user_fraud_summary")
total_users = df_users.count()
users_fraud = df_users.filter(col("total_reviews_fraude") > 0).count()

print(f"\n  Total de Usuários: {total_users:,}")
print(f"  Usuários com Fraude: {users_fraud:,} ({users_fraud/total_users*100:.2f}%)")

# Estatísticas por estabelecimento
df_business = spark.table("yelp_ing.gold_business_fraud_summary")
total_business = df_business.count()
business_fraud = df_business.filter(col("total_reviews_fraude") > 0).count()

print(f"\n  Total de Estabelecimentos: {total_business:,}")
print(f"  Estabelecimentos com Fraude: {business_fraud:,} ({business_fraud/total_business*100:.2f}%)")

# Distribuição temporal
print("\n🕒 HORÁRIOS COM MAIS FRAUDES:")
print("-" * 80)
df_time = spark.table("yelp_ing.gold_reviews_time_distribution")
top_fraud_hours = df_time.orderBy(col("total_fraud").desc()).limit(3).collect()

for i, row in enumerate(top_fraud_hours, 1):
    print(f"  {i}. {row['time_bucket']}: {row['total_fraud']} fraudes de {row['total_reviews']} reviews ({row['total_fraud']/row['total_reviews']*100:.2f}%)")

print("\n" + "=" * 80)
print("🛠️ ARQUITETURA IMPLEMENTADA:")
print("=" * 80)
print("\n  Camadas de Dados:")
print("    ✓ Bronze (já existente): Dados brutos do Yelp")
print("    ✓ Silver: Tabelas de amostra (1000 registros)")
print("    ✓ Silver Refinada: Dados enriquecidos com informações temporais")
print("    ✓ Gold: Tabelas analíticas e métricas de fraude")

print("\n  Lógica de Detecção:")
print("    ✓ Estabelecimento permanentemente fechado (is_open = 0)")
print("    ✓ Review em dia que estabelecimento não abre")
print("    ✓ Review fora do horário de funcionamento")

print("\n" + "=" * 80)
print("🚀 PRÓXIMOS PASSOS PARA ESCALAR:")
print("=" * 80)
print("\n  1. Trocar tabelas de amostra pelas tabelas completas:")
print("     - yelp_ing.silver_reviews_filtered_amostra → yelp_ing.silver_review_filtered")
print("     - yelp_ing.silver_business_amostra → yelp_ing.silver_business")

print("\n  2. Ajustar particionamento para performance:")
print("     - Particionar por review_date na camada Gold")
print("     - Adicionar Z-ORDER em colunas de filtro frequente")

print("\n  3. Adicionar detecções avançadas:")
print("     - Padrões de texto suspeitos (reviews genéricas)")
print("     - Velocidade de publicação (muitos reviews em curto período)")
print("     - Análise de localização geográfica")

print("\n  4. Automatizar com Delta Live Tables (DLT):")
print("     - Pipeline incremental")
print("     - Expectativas de qualidade de dados")
print("     - Monitoramento contínuo")

print("\n  5. Criar Dashboard no Databricks SQL/Genie:")
print("     - KPI Cards com métricas principais")
print("     - Gráficos de distribuição temporal")
print("     - Alertas de usuários e estabelecimentos")

print("\n" + "=" * 80)
print("✅ PIPELINE PRONTO PARA DASHBOARD!")
print("=" * 80)
print("\nUse as queries SQL da ETAPA 7 para criar visualizações no Databricks SQL.")
print("\n")